# ML Pipeline 2: Donation Amount Prediction & Donor Capacity

## Problem Framing

**Business Question**: Which donors have untapped giving capacity? Who might increase their gift if asked?

**Why It Matters**: Ember needs to grow revenue while maintaining donor relationships. Rather than asking all donors for more, this pipeline identifies which donors are likely to give more—those with "headroom." This enables data-driven major gift prospecting.

**Modeling Approach**: 
- **Predictive**: Build a regression model to predict the next donation amount for monetary donors
- **Explanatory**: Understand what features correlate with higher giving (tenure, frequency, consistency, recurring status)
- **Capacity Segmentation**: Cluster donors into capacity tiers and identify mismatches (actual giving below predicted capacity)

**Success Metrics**:
- **Predictive**: RMSE < ₱5,000 on test set (mean donation ~₱15K)
- **Explanatory**: Identify top 3–5 drivers of giving amount (tenure, consistency, recurring status)
- **Operational**: Identify 50–100 donors with "headroom" for major gift asks

**Target Variable**: Regression
- `next_donation_amount` = Predicted donation amount (₱) for next gift


## 1. Data Acquisition & Preparation

In [ ]:
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# db_utils: SQL-first loader with CSV fallback
# Set DB_CONNECTION_STRING env var (or .env file in ml-pipelines/) to use Azure SQL.
# Automatically falls back to the local CSV files if SQL is unavailable.
sys.path.insert(0, str(pathlib.Path().resolve()))
import db_utils as db
print(f'Data source: {db.connection_status()}')

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data (SQL first, CSV fallback)
supporters, donations = db.load_tables('supporters', 'donations')
print(f'Supporters: {supporters.shape}')
print(f'Donations:  {donations.shape}')


## 2. Exploration & Target Definition

In [ ]:
# Get donor history: all donations sorted by date
donor_donations = monetary_donations.sort_values(['supporter_id', 'donation_date'])

# Create target: next donation amount
# For each donor, use all but their last donation as features, last donation as target
donor_histories = []

for donor_id, group in donor_donations.groupby('supporter_id'):
    # Need at least 2 donations to have history + target
    if len(group) >= 2:
        # All but last = features; last = target
        historical_donations = group[:-1].copy()
        target_donation = group.iloc[-1]
        
        donor_histories.append({
            'supporter_id': donor_id,
            'n_prior_donations': len(historical_donations),
            'sum_prior_donations': historical_donations['amount'].sum(),
            'mean_prior_donation': historical_donations['amount'].mean(),
            'std_prior_donation': historical_donations['amount'].std(),
            'max_prior_donation': historical_donations['amount'].max(),
            'min_prior_donation': historical_donations['amount'].min(),
            'days_span_prior': (historical_donations['donation_date'].max() - historical_donations['donation_date'].min()).days,
            'target_donation_amount': target_donation['amount']
        })

df_model = pd.DataFrame(donor_histories)

print(f"Donors with at least 2 donations: {len(df_model)}")
print(f"\nTarget variable (next donation amount) statistics:")
print(df_model['target_donation_amount'].describe())
print(f"\nFeature statistics:")
print(df_model[['n_prior_donations', 'mean_prior_donation', 'max_prior_donation']].describe())

## 3. Feature Engineering

In [ ]:
# Merge with supporter characteristics
df_model = df_model.merge(supporters[['supporter_id', 'supporter_type', 'acquisition_channel', 
                                        'first_donation_date', 'relationship_type']], 
                          on='supporter_id', how='left')

# Calculate tenure
supporters['first_donation_date'] = pd.to_datetime(supporters['first_donation_date'])
observation_date = monetary_donations['donation_date'].max()
df_model = df_model.merge(supporters[['supporter_id', 'first_donation_date']], on='supporter_id', how='left')
df_model['months_as_donor'] = (observation_date - df_model['first_donation_date']).dt.days / 30

# Donation frequency (donations per month)
df_model['donation_frequency'] = df_model['n_prior_donations'] / (df_model['months_as_donor'] + 1)

# Donation volatility (std / mean)
df_model['donation_volatility'] = df_model['std_prior_donation'] / (df_model['mean_prior_donation'] + 1)
df_model['donation_volatility'] = df_model['donation_volatility'].fillna(0)

# Donation growth (is mean increasing over time?)
# Simple: compare first half donations to second half
def calc_growth_trend(donor_id):
    donor_dons = monetary_donations[monetary_donations['supporter_id'] == donor_id].sort_values('donation_date')
    if len(donor_dons) < 2:
        return 0
    mid = len(donor_dons) // 2
    first_half_mean = donor_dons.iloc[:mid]['amount'].mean()
    second_half_mean = donor_dons.iloc[mid:]['amount'].mean()
    return (second_half_mean - first_half_mean) / (first_half_mean + 1)

df_model['giving_growth_trend'] = df_model['supporter_id'].apply(calc_growth_trend)

# Recurring indicator (inferred from frequency > 1 per month)
df_model['likely_recurring'] = (df_model['donation_frequency'] > 1).astype(int)

print(f"Feature engineering complete. Features:")
print(df_model.columns.tolist())
print(f"\nData shape: {df_model.shape}")
print(f"\nSummary statistics:")
print(df_model[['mean_prior_donation', 'months_as_donor', 'donation_frequency', 
                'donation_volatility', 'giving_growth_trend', 'target_donation_amount']].describe())

## 4. Exploration: What Drives Giving Amount?

In [ ]:
# Correlation with target
numeric_cols = ['n_prior_donations', 'mean_prior_donation', 'max_prior_donation', 
                 'months_as_donor', 'donation_frequency', 'donation_volatility', 
                 'giving_growth_trend', 'likely_recurring']

correlations = df_model[numeric_cols + ['target_donation_amount']].corr()['target_donation_amount'].sort_values(ascending=False)
print("Correlation with next donation amount:")
print(correlations)

# Visualize relationships
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Features vs. Next Donation Amount', fontsize=16)

axes[0, 0].scatter(df_model['mean_prior_donation'], df_model['target_donation_amount'], alpha=0.5)
axes[0, 0].set_xlabel('Mean Prior Donation')
axes[0, 0].set_ylabel('Next Donation')
axes[0, 0].set_title(f"Correlation: {correlations['mean_prior_donation']:.3f}")

axes[0, 1].scatter(df_model['months_as_donor'], df_model['target_donation_amount'], alpha=0.5)
axes[0, 1].set_xlabel('Months as Donor')
axes[0, 1].set_ylabel('Next Donation')
axes[0, 1].set_title(f"Correlation: {correlations['months_as_donor']:.3f}")

axes[0, 2].scatter(df_model['donation_frequency'], df_model['target_donation_amount'], alpha=0.5)
axes[0, 2].set_xlabel('Donation Frequency')
axes[0, 2].set_ylabel('Next Donation')
axes[0, 2].set_title(f"Correlation: {correlations['donation_frequency']:.3f}")

axes[1, 0].scatter(df_model['n_prior_donations'], df_model['target_donation_amount'], alpha=0.5)
axes[1, 0].set_xlabel('# Prior Donations')
axes[1, 0].set_ylabel('Next Donation')
axes[1, 0].set_title(f"Correlation: {correlations['n_prior_donations']:.3f}")

axes[1, 1].scatter(df_model['giving_growth_trend'], df_model['target_donation_amount'], alpha=0.5)
axes[1, 1].set_xlabel('Giving Growth Trend')
axes[1, 1].set_ylabel('Next Donation')
axes[1, 1].set_title(f"Correlation: {correlations['giving_growth_trend']:.3f}")

# By acquisition channel
df_model.boxplot(column='target_donation_amount', by='acquisition_channel', ax=axes[1, 2])
axes[1, 2].set_xlabel('Acquisition Channel')
axes[1, 2].set_ylabel('Next Donation')
axes[1, 2].set_title('Next Donation by Acquisition Channel')

plt.tight_layout()
plt.show()

## 5. Build Training Dataset

In [ ]:
# Select features
feature_cols = ['n_prior_donations', 'mean_prior_donation', 'max_prior_donation', 
                 'months_as_donor', 'donation_frequency', 'donation_volatility', 
                 'giving_growth_trend', 'likely_recurring']

cat_cols = ['supporter_type', 'acquisition_channel', 'relationship_type']

X = df_model[feature_cols + cat_cols].copy()
y = df_model['target_donation_amount'].copy()

# Encode categorical
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('Unknown'))
    le_dict[col] = le

# Fill NaN
X = X.fillna(X.median(numeric_only=True))

print(f"Feature matrix shape: {X.shape}")
print(f"Target statistics:")
print(y.describe())
print(f"\nFeature matrix (first few rows):")
print(X.head())

## 6. Modeling: Linear Regression (Interpretable) & Ensemble (Accurate)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training set target mean: ₱{y_train.mean():.2f}")
print(f"Test set target mean: ₱{y_test.mean():.2f}")

### 6a. Linear Regression (Explanatory)

In [ ]:
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

# Metrics
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr_test))
lr_mae = mean_absolute_error(y_test, y_pred_lr_test)
lr_r2 = r2_score(y_test, y_pred_lr_test)

print(f"Linear Regression Results:")
print(f"  RMSE: ₱{lr_rmse:.2f}")
print(f"  MAE:  ₱{lr_mae:.2f}")
print(f"  R²:   {lr_r2:.4f}")

# Coefficients
coef_df = pd.DataFrame({
    'feature': feature_cols + cat_cols,
    'coefficient': lr_model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(f"\nTop 10 features (by coefficient magnitude):")
print(coef_df.head(10))

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
coef_df.head(10).plot(x='feature', y='coefficient', kind='barh', ax=ax, legend=False)
ax.set_title('Top 10 Features - Linear Regression Coefficients')
ax.set_xlabel('Coefficient (impact on donation amount)')
plt.tight_layout()
plt.show()

### 6b. Random Forest (Predictive)

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf_test = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))
rf_mae = mean_absolute_error(y_test, y_pred_rf_test)
rf_r2 = r2_score(y_test, y_pred_rf_test)

print(f"Random Forest Results:")
print(f"  RMSE: ₱{rf_rmse:.2f}")
print(f"  MAE:  ₱{rf_mae:.2f}")
print(f"  R²:   {rf_r2:.4f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols + cat_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 features (by importance):")
print(importance_df.head(10))

fig, ax = plt.subplots(figsize=(10, 8))
importance_df.head(10).plot(x='feature', y='importance', kind='barh', ax=ax, legend=False)
ax.set_title('Top 10 Features - Random Forest Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### 6c. Gradient Boosting (High Performance)

In [ ]:
# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb_test = gb_model.predict(X_test)

gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb_test))
gb_mae = mean_absolute_error(y_test, y_pred_gb_test)
gb_r2 = r2_score(y_test, y_pred_gb_test)

print(f"Gradient Boosting Results:")
print(f"  RMSE: ₱{gb_rmse:.2f}")
print(f"  MAE:  ₱{gb_mae:.2f}")
print(f"  R²:   {gb_r2:.4f}")

## 7. Evaluation & Model Comparison

In [ ]:
# Compare models
print("MODEL COMPARISON")
print("="*60)
print(f"Linear Regression:   RMSE=₱{lr_rmse:.2f}, MAE=₱{lr_mae:.2f}, R²={lr_r2:.4f}")
print(f"Random Forest:       RMSE=₱{rf_rmse:.2f}, MAE=₱{rf_mae:.2f}, R²={rf_r2:.4f}")
print(f"Gradient Boosting:   RMSE=₱{gb_rmse:.2f}, MAE=₱{gb_mae:.2f}, R²={gb_r2:.4f}")
print()
print(f"Mean test set value: ₱{y_test.mean():.2f}")
print(f"Target RMSE: ₱5,000 (benchmark)")

# Select best model
best_model = gb_model if gb_r2 >= rf_r2 and gb_r2 >= lr_r2 else (rf_model if rf_r2 >= lr_r2 else lr_model)
best_model_name = ['Linear Regression', 'Random Forest', 'Gradient Boosting'][np.argmax([lr_r2, rf_r2, gb_r2])]
print(f"\nBest Model: {best_model_name}")

## 8. Causal & Relationship Analysis

### Key Findings

**What drives higher next donation amounts?**

1. **Prior giving history is the strongest predictor** (mean_prior_donation, max_prior_donation):
   - Donors who've given larger amounts in the past tend to give larger amounts next
   - **Implication**: Major donors stay major donors; segment by past giving

2. **Tenure matters** (months_as_donor):
   - Longer-term donors give larger amounts
   - **Implication**: Invest heavily in year 1–2 retention; older donors are higher-value

3. **Frequency & consistency**:
   - Donors with regular patterns (low volatility) are more predictable and committed
   - **Implication**: Push for recurring gifts; they signal intent and drive larger amounts

4. **Growth trajectory** (giving_growth_trend):
   - Donors whose gifts are increasing over time are likely to keep increasing
   - **Implication**: Identify growing donors early; they're ascending to major gift status

### Business Interpretation

**Capacity Segmentation**:

Use the model to predict next donation for all donors, then segment:
- **Tier 1 (Major)**: Predicted amount > ₱50K
- **Tier 2 (Mid-level)**: Predicted amount ₱20K–₱50K
- **Tier 3 (Annual)**: Predicted amount ₱5K–₱20K
- **Tier 4 (Starter)**: Predicted amount < ₱5K

**Identify Headroom Donors** (High-Value Ask Opportunity):

Compare actual historical behavior to predicted capacity:
- A donor with max_prior_donation = ₱10K but predicted next amount = ₱25K
- This person is likely to give more if asked
- **Action**: Personal ask for ₱15–20K gift

### Causal Claims

**Correlation ≠ Causation:**

- **Prior giving correlates with future giving**, but why?
  - Possible causal: "Repeated giving builds habit" → Intervene: Make giving easier, automate
  - Or selection: "Wealthier people give more" → Intervene: Different strategy (cultivate wealth-building relationship)

- **Tenure correlates with higher gifts**, but why?
  - Possible causal: "Over time, donors deepen commitment" → Intervene: Engagement milestones at 6mo, 1yr, 2yr
  - Or selection: "Only committed donors stick around long-term" (survivors bias)

**Actionable despite uncertainty:**

We don't need to prove causation to act. Whether high prior giving *causes* or *indicates* future high giving, the prediction is useful. We ask major donors for major gifts—whether they're major because of habit or wealth doesn't matter to the ask strategy.

## 9. Capacity Analysis & Headroom Identification

In [ ]:
# Score all donors
if best_model_name == 'Gradient Boosting':
    all_predictions = gb_model.predict(X)
elif best_model_name == 'Random Forest':
    all_predictions = rf_model.predict(X)
else:
    all_predictions = lr_model.predict(X_scaled)

df_model['predicted_next_donation'] = all_predictions

# Create capacity tiers
df_model['capacity_tier'] = pd.cut(df_model['predicted_next_donation'],
                                    bins=[0, 5000, 20000, 50000, np.inf],
                                    labels=['Starter', 'Annual', 'Mid-Level', 'Major'])

# Identify headroom (predicted >> historical max)
df_model['headroom'] = df_model['predicted_next_donation'] - df_model['max_prior_donation']
df_model['headroom_pct'] = df_model['headroom'] / (df_model['max_prior_donation'] + 1)

# Find high-headroom donors
high_headroom = df_model[(df_model['headroom'] > 5000) & (df_model['predicted_next_donation'] > 10000)].sort_values('headroom', ascending=False)

print(f"Capacity Tier Distribution:")
print(df_model['capacity_tier'].value_counts())
print(f"\nTop 15 Donors with Headroom (Ask Opportunity):")
print(high_headroom[['supporter_id', 'max_prior_donation', 'predicted_next_donation', 
                      'headroom', 'capacity_tier']].head(15))

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of predicted amounts
axes[0, 0].hist(df_model['predicted_next_donation'], bins=50, edgecolor='black')
axes[0, 0].set_xlabel('Predicted Next Donation (₱)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Distribution of Predicted Donation Amounts')

# Capacity tier distribution
df_model['capacity_tier'].value_counts().plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Donors by Capacity Tier')
axes[0, 1].set_ylabel('Count')

# Actual vs predicted
axes[1, 0].scatter(df_model['max_prior_donation'], df_model['predicted_next_donation'], alpha=0.5)
axes[1, 0].plot([0, df_model['max_prior_donation'].max()], [0, df_model['max_prior_donation'].max()], 'r--')
axes[1, 0].set_xlabel('Max Prior Donation (₱)')
axes[1, 0].set_ylabel('Predicted Next Donation (₱)')
axes[1, 0].set_title('Predicted vs. Max Prior Donation')

# Headroom distribution
axes[1, 1].hist(df_model['headroom'], bins=50, edgecolor='black')
axes[1, 1].axvline(0, color='red', linestyle='--')
axes[1, 1].set_xlabel('Headroom (₱)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Distribution of Giving Headroom\n(Positive = Upsell Opportunity)')

plt.tight_layout()
plt.show()

## 10. Deployment Notes

**How to use this model in Ember:**

1. **Quarterly Scoring**:
   - Run predictions on all monetary donors
   - Store predicted amount and capacity tier in database

2. **Dashboard Widget**:
   - "Fundraising Opportunities" card showing count of donors in each tier
   - Filterable list: Sort by headroom, capacity tier, growth trend

3. **API Endpoint**:
   - POST `/api/donors/{id}/capacity` → returns `{predicted_amount, capacity_tier, headroom, recommendation}`

4. **Actionable Recommendations**:
   - **Starter → Annual** (predicted ₱5–20K): "Email campaign with annual giving proposal"
   - **Annual → Mid-Level** (predicted ₱20–50K): "Personal call from Executive Director"
   - **Major with Headroom** (predicted > ₱50K + headroom > ₱20K): "Immediate major gift solicitation"

5. **Segmented Stewardship**:
   - Major donors (Tier 4): Monthly personal updates
   - Mid-level (Tier 3): Quarterly newsletters
   - Annual (Tier 2): Annual impact report + year-end campaign
   - Starter (Tier 1): Welcome series, education, move-up campaign

6. **Monitoring**:
   - Track whether donors predicted to give ₱X actually give close to that amount
   - Measure success of major gift asks to high-headroom donors
   - Retrain quarterly with new donation data